# Reward transform and task termination

This notebook shows the two `EnvConfig` callables that shape a rollout, plus the episode-count timeout.

| Knob | Role |
|------|------|
| `reward_transform` | Rewrites each step reward (including reset frames) before the step output and before metrics. |
| `terminate_task` | On an episode-end step, a truthy return emits `task_done=1` (task **terminated**). |
| `max_task_episodes` | After this many episodes, emit `task_done=2` (task **truncated**). |

Both callables receive the same keyword arguments: `step_index`, `episode_index`, `state`, `action`, `reward`, `done`, `next_state`. `done` is `episode_done` (`0` / `1` / `2`). `reward` in `terminate_task` is already the post-`reward_transform` value. Accept unused fields with `**kwargs`.

A reset frame is `step_index == 0`. A task-start reset is also `episode_index == 0`.

If `terminate_task` and `max_task_episodes` would fire on the same step, `task_done=1` wins.

The env below ends on the first step (`episode_done=1`) so every boundary is visible in a short loop.

In [6]:
import gymnasium as gym

from mouse_gym import EnvConfig, make_env


class OneStepEnv(gym.Env):
    """Reset obs `0`; one step returns obs `1`, reward `1.0`, terminated."""

    observation_space = gym.spaces.Discrete(2)
    action_space = gym.spaces.Discrete(2)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return 0, {}

    def step(self, action):
        return 1, 1.0, True, False, {}


def rollout(env, steps: int) -> None:
    for _ in range(steps):
        output = env.step(env.sample_random_input())
        print(
            f"task={output['task_index']} ep={output['episode_index']} "
            f"step={int(output['step_index'])} reward={float(output['reward']):.1f} "
            f"episode_done={int(output['episode_done'])} "
            f"task_done={int(output['task_done'])}"
        )

## 1. `reward_transform`

Called on every frame. Reset frames pass `step_index=0`, `reward=0.0`, and `action=None`. A new task is `episode_index=0` on that same frame; an episode reset mid-task has `episode_index > 0`. Env steps pass the raw Gymnasium reward. The returned value is written to the step output and accumulated into episode/task metrics.

In [7]:
def reward_transform(*, step_index, episode_index, reward, **_kwargs):
    if step_index == 0 and episode_index == 0:
        return -0.5
    if step_index == 0:
        return -0.1
    return reward * 10.0


env = make_env(
    EnvConfig(
        seed=0,
        env_fn=OneStepEnv,
        reward_transform=reward_transform,
        max_task_episodes=2,
    )
)
rollout(env, steps=6)
print("episode returns:", env.metrics.episode_cum_rewards)
print("task returns   :", env.metrics.task_cum_rewards)
env.close()

task=0 ep=0 step=0 reward=-0.5 episode_done=0 task_done=0
task=0 ep=0 step=1 reward=10.0 episode_done=1 task_done=0
task=0 ep=1 step=0 reward=-0.1 episode_done=0 task_done=0
task=0 ep=1 step=1 reward=10.0 episode_done=1 task_done=2
task=1 ep=0 step=0 reward=-0.5 episode_done=0 task_done=0
task=1 ep=0 step=1 reward=10.0 episode_done=1 task_done=0
episode returns: [9.5, 9.9, 9.5]
task returns   : [19.4]


Each episode is reset reward `-0.1` plus one env step `10.0`, so the episode return is `9.9`. Two episodes fill a task (`max_task_episodes=2`), so the task return is `19.8`. On the terminal step of episode 1 you see `task_done=2`.

## 2. `terminate_task` → `task_done=1`

Invoked only on episode-end steps. Return `True` to terminate the task. Here the task ends as soon as an episode terminates (`done == 1`).

In [8]:
def terminate_task(*, done, **_kwargs):
    return done == 1


env = make_env(
    EnvConfig(
        seed=0,
        env_fn=OneStepEnv,
        terminate_task=terminate_task,
    )
)
rollout(env, steps=6)
print("task returns:", env.metrics.task_cum_rewards)
env.close()

task=0 ep=0 step=0 reward=0.0 episode_done=0 task_done=0
task=0 ep=0 step=1 reward=1.0 episode_done=1 task_done=1
task=1 ep=0 step=0 reward=0.0 episode_done=0 task_done=0
task=1 ep=0 step=1 reward=1.0 episode_done=1 task_done=1
task=2 ep=0 step=0 reward=0.0 episode_done=0 task_done=0
task=2 ep=0 step=1 reward=1.0 episode_done=1 task_done=1
task returns: [1.0, 1.0, 1.0]


Every episode-end step has `task_done=1`. The next `step()` is a reset frame that starts a new task (`task_index` increments, `episode_index` goes back to `0`).

## 3. `max_task_episodes` → `task_done=2`

Without `terminate_task`, the only automatic task boundary is the episode budget.

In [9]:
env = make_env(
    EnvConfig(
        seed=0,
        env_fn=OneStepEnv,
        max_task_episodes=2,
    )
)
rollout(env, steps=8)
env.close()

task=0 ep=0 step=0 reward=0.0 episode_done=0 task_done=0
task=0 ep=0 step=1 reward=1.0 episode_done=1 task_done=0
task=0 ep=1 step=0 reward=0.0 episode_done=0 task_done=0
task=0 ep=1 step=1 reward=1.0 episode_done=1 task_done=2
task=1 ep=0 step=0 reward=0.0 episode_done=0 task_done=0
task=1 ep=0 step=1 reward=1.0 episode_done=1 task_done=0
task=1 ep=1 step=0 reward=0.0 episode_done=0 task_done=0
task=1 ep=1 step=1 reward=1.0 episode_done=1 task_done=2


Episode 0 of each task ends with `task_done=0`. Episode 1 ends with `task_done=2`.

## 4. Both: termination wins

When `terminate_task` returns true on the same step that would hit `max_task_episodes`, the output is `task_done=1`, not `2`.

In [10]:
env = make_env(
    EnvConfig(
        seed=0,
        env_fn=OneStepEnv,
        max_task_episodes=1,
        terminate_task=lambda done, **_: done == 1,
        reward_transform=lambda step_index, reward, **_: (
            -0.1 if step_index == 0 else reward * 10.0
        ),
    )
)
rollout(env, steps=6)
print("episode returns:", env.metrics.episode_cum_rewards)
print("task returns   :", env.metrics.task_cum_rewards)
env.close()

task=0 ep=0 step=0 reward=-0.1 episode_done=0 task_done=0
task=0 ep=0 step=1 reward=10.0 episode_done=1 task_done=1
task=1 ep=0 step=0 reward=-0.1 episode_done=0 task_done=0
task=1 ep=0 step=1 reward=10.0 episode_done=1 task_done=1
task=2 ep=0 step=0 reward=-0.1 episode_done=0 task_done=0
task=2 ep=0 step=1 reward=10.0 episode_done=1 task_done=1
episode returns: [9.9, 9.9, 9.9]
task returns   : [9.9, 9.9, 9.9]
